In [ ]:
!pip uninstall -y scikit-learn imbalanced-learn
!pip install scikit-learn==1.3.2 imbalanced-learn==0.11.0 --no-cache-dir --force-reinstall


In [ ]:
import sklearn
import imblearn

print(sklearn.__version__)
print(imblearn.__version__)

from imblearn.over_sampling import SMOTE
print("SMOTE imported successfully!")

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
import os
import matplotlib.pyplot as plt  # Import matplotlib to plot images
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Check the files in the /kaggle/input directory to find your dataset
input_dir = '/kaggle/input/'
print("Files in Kaggle Input Directory:")
for root, dirs, files in os.walk(input_dir):
    print(f"Root: {root}")
    print(f"Dirs: {dirs}")
    print(f"Files: {files}")


In [ ]:
# Define paths
original_data_dir = '/kaggle/input/datasets/dpsiam324/chili-enhanced-images'
base_dir = '/kaggle/working/chili_split_data'
classes = os.listdir(original_data_dir)
print(classes)

In [ ]:
# Step 2: Create split folders again
for cls in classes:
    os.makedirs(os.path.join(base_dir, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(base_dir, 'test', cls), exist_ok=True)

    # Step 3: Filter image files only
    files = [f for f in os.listdir(os.path.join(original_data_dir, cls))
             if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    # Step 4: Reproducible shuffle
    random.seed(42)
    random.shuffle(files)

    # Step 5: Split logic
    total = len(files)
    train_count = int(0.80 * total)
    for i, file in enumerate(files):
        src = os.path.join(original_data_dir, cls, file)
        if i < train_count:
            dst = os.path.join(base_dir, 'train', cls, file)
        else:
            dst = os.path.join(base_dir, 'test', cls, file)
        shutil.copy(src, dst)  # or use shutil.move(src, dst) if splitting permanently

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

data_path = "/kaggle/working/chili_split_data"

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(data_path, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(data_path, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

In [ ]:
def extract_and_concatenate_features(generator, models, num_samples):
    features_list = [[] for _ in models]
    labels = []

    for images, lbls in generator:
        # Extract features for each model
        for i, model in enumerate(models):
            batch_features = model.predict(images, verbose=0)
            features_list[i].extend(batch_features)

        labels.extend(lbls)
        if len(labels) >= num_samples:
            break

    # Stack features and labels
    features = [np.array(feats[:num_samples]) for feats in features_list]
    concatenated = np.concatenate(features, axis=1)  # concat along feature dimension
    return concatenated, np.array(labels[:num_samples], dtype=int)


In [ ]:
models = [feature_model1, feature_model2, feature_model3]

X_train, y_train = extract_and_concatenate_features(train_generator, models, train_generator.samples)
X_test, y_test   = extract_and_concatenate_features(test_generator, models, test_generator.samples)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Step 1: Get class name mapping from the generator
class_indices = train_generator.class_indices
inv_class_indices = {v: k for k, v in class_indices.items()}

# Step 2: Convert numeric labels (e.g., 0,1,2) to class names (e.g., "Healthy", "Rust", etc.)
y_train_names = [inv_class_indices[i] for i in y_train]

# Step 3: Run PCA to reduce dimensionality to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

# Step 4: Plot with seaborn
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=y_train_names,
    palette='Set2',
    s=50,
    alpha=0.9
)

plt.title("PCA Projection of Combined Features (by Disease Class)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Disease Class", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error,
    precision_score, recall_score, f1_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

class_indices = train_generator.class_indices
inv_class_indices = {v: k for k, v in class_indices.items()}
class_names = [inv_class_indices[i] for i in sorted(inv_class_indices)]


xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_train_resampled)),
    eval_metric='mlogloss',
    use_label_encoder=False,
    learning_rate=0.05,
    n_estimators=150,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(np.unique(y_train_resampled)),
    learning_rate=0.05,
    n_estimators=150,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

et_model = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=25,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

meta_learner = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    C=0.5,
    max_iter=1000
)

ChiliFusionNet = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('rf', rf_model),
        ('et', et_model)
    ],
    final_estimator=meta_learner,
    passthrough=True,
    cv=5,
    n_jobs=-1
)

ChiliFusionNet.fit(X_train_resampled, y_train_resampled)

y_pred_ensemble = ChiliFusionNet.predict(X_test)
y_proba_ensemble = ChiliFusionNet.predict_proba(X_test)

cm = confusion_matrix(y_test, y_pred_ensemble)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title("ChiliFusionNet - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

print("📊 Classification Report (ChiliFusionNet):")
print(classification_report(y_test, y_pred_ensemble, target_names=class_names))

accuracy = accuracy_score(y_test, y_pred_ensemble)
error_rate = 1 - accuracy
mae = mean_absolute_error(y_test, y_pred_ensemble)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ensemble))
rae = np.sum(np.abs(y_test - y_pred_ensemble)) / np.sum(np.abs(y_test - np.mean(y_test)))

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Misclassification Rate: {error_rate:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"RAE: {rae:.4f}")

print(f"\nPrecision (Macro): {precision_score(y_test, y_pred_ensemble, average='macro'):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred_ensemble, average='macro'):.4f}")
print(f"F1 Score (Macro): {f1_score(y_test, y_pred_ensemble, average='macro'):.4f}")

print(f"\nPrecision (Weighted): {precision_score(y_test, y_pred_ensemble, average='weighted'):.4f}")
print(f"Recall (Weighted): {recall_score(y_test, y_pred_ensemble, average='weighted'):.4f}")
print(f"F1 Score (Weighted): {f1_score(y_test, y_pred_ensemble, average='weighted'):.4f}")

n_classes = len(class_names)
y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba_ensemble[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
colors = ['blue', 'green', 'orange', 'red', 'purple', 'brown']
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], color=colors[i % len(colors)],
             label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.title("ROC Curve - Optimized ChiliFusionNet")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
y_train_pred = ChiliFusionNet.predict(X_train_resampled)

train_accuracy = accuracy_score(y_train_resampled, y_train_pred)
print(f"Training Accuracy: {train_accuracy:.4f}")
